In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.utils.function_calling import convert_to_openai_function

In [2]:
import json

@tool
def execute_python_code(code: str) -> str:
    """
    Execute Python code safely using eval.
    
    Args:
        code: Python code to execute (simple expressions only)
    
    Returns:
        String containing the result of the code execution
    """
    try:
        import sys
        from io import StringIO
        
        # Capture stdout
        old_stdout = sys.stdout
        sys.stdout = captured_output = StringIO()
        
        # Create a safe execution environment
        safe_globals = {"__builtins__": __builtins__}
        safe_locals = {}
        
        # Execute the code
        exec(code, safe_globals, safe_locals)
        
        # Get the output
        output = captured_output.getvalue()
        
        # Restore stdout
        sys.stdout = old_stdout
        
        if output.strip():
            return f"The result of the code execution is:\n\n```\n{output.strip()}\n```"
        else:
            # If no output, try to get the last expression value
            try:
                result = eval(code, safe_globals, safe_locals)
                return f"The result of the code execution is:\n\n```\n{result}\n```"
            except:
                return "Code executed successfully (no output)"
                
    except Exception as e:
        return f"Error executing code: {str(e)}"

# # Print the schema of this tool
print(json.dumps(execute_python_code.args_schema.model_json_schema()))

{"description": "Execute Python code safely using eval.\n\nArgs:\n    code: Python code to execute (simple expressions only)\n\nReturns:\n    String containing the result of the code execution", "properties": {"code": {"title": "Code", "type": "string"}}, "required": ["code"], "title": "execute_python_code", "type": "object"}


In [3]:
convert_to_openai_function(execute_python_code.args_schema.model_json_schema())

{'name': 'execute_python_code',
 'description': 'Execute Python code safely using eval.\n\nArgs:\n    code: Python code to execute (simple expressions only)\n\nReturns:\n    String containing the result of the code execution',
 'parameters': {'properties': {'code': {'title': 'Code', 'type': 'string'}},
  'required': ['code'],
  'type': 'object'}}

In [23]:
import json

from langchain_anthropic import ChatAnthropic
from langchain_ollama import ChatOllama

reasoning_llm = ChatAnthropic(
    model="claude-sonnet-4-0",
    max_tokens=5000,
    model_kwargs={"thinking": {"type": "enabled", "budget_tokens": 2000}},
)

# Requires Ollama to be installed
# reasoning_llm = ChatOllama(
#     model="qwen3:0.6b",
#     max_tokens=None,
# )


/Users/sinanozdemir/Teaching/Pearson/oreilly-ai-agents/myenv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3607: UserWarning: Parameters {'thinking'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [24]:
response = reasoning_llm.invoke("What is the cube root of 27?")
print(json.dumps(response.content, indent=2))

response

[
  {
    "signature": "ErUDCkYICBgCKkCEJ85RLcErVP7DmDG5aIGK1MxgZc2dRUV4OzIDmCtaIVBlK7RyyJpWoS3fZSUCE7D0dlqZsSNegSRNLsD77FNFEgw/L8CWZF6kS8dhkNgaDKuX9Yu5OnKACffjbCIweCmm2bpUYWO+bfPFhoTRb3rwCtYFGx7DVSGgje3Lf3T7Lj+jxjfwUK1Y/MvmQzdNKpwCP2I2wsAv5SFhxZsqAFZ1wrCE1RbLEhO/sGZfpI2dFEXqMl5Dq1NQDDXM0uHYiuBJKkdcR3HcuWS+5V4dernVPrz0oz8PmMNL/ZPYzPMDRhVRni0aM7AMtFkRJd0d+SY4PpqN3kNJvWor6jFNKbEOMNuJBndxTwxz2NE0O14ZieTGZ58OlxYflseuHNeHavB87ftl0o9cxAoTkUjk9KFOwehmWZ80PkKogYpFzdpmDNyHULD2719f6ofzw8TdKTEg2RvU+xuSCDbhOn133tmp2M+itFftxgxN2UEC+tOIGODxI2pJEZ7k8ePIGdRClzbCIu5KVE01wie/2RdG6gRsDgKtuvSUsxbhbw/UAQbAsbQYQX28MeQzwgw2H2IYAQ==",
    "thinking": "I need to find the cube root of 27. The cube root of a number x is the number that when multiplied by itself three times equals x.\n\nSo I'm looking for a number n such that n\u00b3 = 27.\n\nLet me think about this:\n- 1\u00b3 = 1\n- 2\u00b3 = 8\n- 3\u00b3 = 27\n\nSo the cube root of 27 is 3.",
    "type": "thinking"
  },
  {
    "text": "The cube root of 27 is 

AIMessage(content=[{'signature': 'ErUDCkYICBgCKkCEJ85RLcErVP7DmDG5aIGK1MxgZc2dRUV4OzIDmCtaIVBlK7RyyJpWoS3fZSUCE7D0dlqZsSNegSRNLsD77FNFEgw/L8CWZF6kS8dhkNgaDKuX9Yu5OnKACffjbCIweCmm2bpUYWO+bfPFhoTRb3rwCtYFGx7DVSGgje3Lf3T7Lj+jxjfwUK1Y/MvmQzdNKpwCP2I2wsAv5SFhxZsqAFZ1wrCE1RbLEhO/sGZfpI2dFEXqMl5Dq1NQDDXM0uHYiuBJKkdcR3HcuWS+5V4dernVPrz0oz8PmMNL/ZPYzPMDRhVRni0aM7AMtFkRJd0d+SY4PpqN3kNJvWor6jFNKbEOMNuJBndxTwxz2NE0O14ZieTGZ58OlxYflseuHNeHavB87ftl0o9cxAoTkUjk9KFOwehmWZ80PkKogYpFzdpmDNyHULD2719f6ofzw8TdKTEg2RvU+xuSCDbhOn133tmp2M+itFftxgxN2UEC+tOIGODxI2pJEZ7k8ePIGdRClzbCIu5KVE01wie/2RdG6gRsDgKtuvSUsxbhbw/UAQbAsbQYQX28MeQzwgw2H2IYAQ==', 'thinking': "I need to find the cube root of 27. The cube root of a number x is the number that when multiplied by itself three times equals x.\n\nSo I'm looking for a number n such that n³ = 27.\n\nLet me think about this:\n- 1³ = 1\n- 2³ = 8\n- 3³ = 27\n\nSo the cube root of 27 is 3.", 'type': 'thinking'}, {'text': 'The cube root of 27 is 3.\n\nThis is because 3 × 3 

In [25]:
model_name = reasoning_llm.model if hasattr(reasoning_llm, "model") else reasoning_llm.model_name
print(f"Using model: {model_name}")

Using model: claude-sonnet-4-0


In [26]:
# Create the React agent
simple_reasoning_agent = create_agent(
    reasoning_llm,
    tools=[execute_python_code],
    checkpointer=MemorySaver()
)

print(f"🚀 Testing Simple Reasoning Agent with {model_name}")

# Test it
try:
    result = simple_reasoning_agent.invoke(
        {"messages": [HumanMessage(content="Calculate 25 * 37 + 100 using Python code and once you do, write another code snippet that multiples it by 1,454,233")]},
        config={"configurable": {"thread_id": "simple-test"}}
    )
    print("✅ Simple agent completed successfully!")
    for msg in result["messages"]:
        print("=" * 60)
        print(f"{type(msg)}")
        reasoning = None
        if hasattr(msg, "additional_kwargs") and "reasoning_content" in getattr(msg, "additional_kwargs", {}):
            reasoning = msg.additional_kwargs["reasoning_content"]
        elif hasattr(msg, "reasoning") and getattr(msg, "reasoning"):
            reasoning = getattr(msg, "reasoning")
        if reasoning:
            print(f"  [REASONING]: {reasoning}")

        # Print Anthropic-style content if present
        if hasattr(msg, "content") and isinstance(msg.content, list):
            for part in msg.content:
                if isinstance(part, dict):
                    part_type = part.get("type")
                    if part_type == "thinking":
                        print(f"  [THINKING]: {part.get('thinking')}")
                    elif part_type == "text":
                        print(f"  [TEXT]: {part.get('text')}")
                    elif part_type == "tool_use":
                        print(f"  [TOOL USE]: {part}")
                    else:
                        print(f"  [PART-{part_type}]: {part}")
                else:
                    print(f"  [PART]: {part}")
        elif hasattr(msg, "content") and isinstance(msg.content, str):
            print(f"  [TEXT]: {msg.content}")

        # Print Groq-style tool calls if present
        # (tool_calls is a list of dicts, each with keys: name, args, id, type)
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"  [TOOL CALL]:")
                print(f"    id: {tool_call.get('id')}")
                print(f"    name: {tool_call.get('name')}")
                print(f"    type: {tool_call.get('type')}")
                print(f"    args: {tool_call.get('args')}")
    print("=" * 60)

except Exception as e:
    print(f"❌ Error with simple agent: {e}")
    print(f"Error type: {type(e).__name__}")


🚀 Testing Simple Reasoning Agent with claude-sonnet-4-0
✅ Simple agent completed successfully!
<class 'langchain_core.messages.human.HumanMessage'>
  [TEXT]: Calculate 25 * 37 + 100 using Python code and once you do, write another code snippet that multiples it by 1,454,233
<class 'langchain_core.messages.ai.AIMessage'>
  [THINKING]: The user wants me to:
1. Calculate 25 * 37 + 100 using Python code
2. Then take that result and multiply it by 1,454,233 in another code snippet

I'll use the execute_python_code function to do this in two steps.
  [TEXT]: I'll help you calculate this step by step using Python code.

First, let me calculate 25 * 37 + 100:
  [TOOL USE]: {'id': 'toolu_01C4kHZBM8wbt6XrDbvtwC5Y', 'input': {'code': '25 * 37 + 100'}, 'name': 'execute_python_code', 'type': 'tool_use'}
  [TOOL CALL]:
    id: toolu_01C4kHZBM8wbt6XrDbvtwC5Y
    name: execute_python_code
    type: tool_call
    args: {'code': '25 * 37 + 100'}
<class 'langchain_core.messages.tool.ToolMessage'>
  [TEXT

In [9]:
from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["serpapi"])

In [10]:
tools

[Tool(name='Search', description='A search engine. Useful for when you need to answer questions about current events. Input should be a search query.', func=<bound method SerpAPIWrapper.run of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='dbfd8b5b815e486ed203c0401c905bbae1a19592b2d785923e4b674f2dc270df', aiosession=None)>, coroutine=<bound method SerpAPIWrapper.arun of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='dbfd8b5b815e486ed203c0401c905bbae1a19592b2d785923e4b674f2dc270df', aiosession=None)>)]

In [11]:
# Set up the ChatOllama model with qwen3:0.6b
reasoning_llm = ChatOllama(
    model="qwen3:0.6b",
    max_tokens=None,
)

# Create a React agent that can search Google
web_search_agent = create_agent(
    reasoning_llm,
    tools=tools,
    checkpointer=MemorySaver()
)

print(f"🔍 Google Search Agent with qwen3:0.6b is ready!")
print("You can now ask me to search for anything on the web.")


🔍 Google Search Agent with qwen3:0.6b is ready!
You can now ask me to search for anything on the web.


In [12]:
# Test the agent
try:
    result = web_search_agent.invoke(
        {"messages": [HumanMessage(content="Who is Sinan Ozdemir in AI?")]},
        config={"configurable": {"thread_id": "web-search-test"}}
    )
    
    print("\n" + "="*60)
    print("🎉 Google Search Agent Test Results:")
    print("="*60)
    
    for msg in result["messages"]:
        if hasattr(msg, 'content') and isinstance(msg.content, str) and msg.content.strip():
            msg_type = type(msg).__name__
            print(f"\n[{msg_type}]:")
            print(msg.content)
            
        for ai_responses in result['messages'][1:]:
            # Print all AI responses after the first message (which is usually the HumanMessage)
            if hasattr(ai_responses, 'content') and isinstance(ai_responses.content, str) and ai_responses.content.strip():
                msg_type = type(ai_responses).__name__
                print(f"\n[{msg_type}]:")
                print(ai_responses.content)


except Exception as e:
    print(f"❌ Error with Google search agent: {e}")
    print("Make sure Ollama is running and the qwen3:0.6b model is installed:")
    print("  ollama pull qwen3:0.6b")



🎉 Google Search Agent Test Results:

[HumanMessage]:
Who is Sinan Ozdemir in AI?

[AIMessage]:
<think>
Okay, the user is asking, "Who is Sinan Ozdemir in AI?" Let me break this down. First, I need to determine if there's a specific AI-related question they're referring to. The name Sinan Ozdemir is associated with AI, so maybe they want information about his role or contributions to AI technology.

I should check if there's a function available to search for this information. The provided tool is a search engine that takes a query. The function requires an argument, which in this case would be the search query. Since the user is asking about Sinan Ozdemir's role in AI, the best approach is to use the search function with the query "Sinan Ozdemir in AI" to get the relevant details. 

I need to make sure the function call is correctly formatted in JSON within the tool_call tags. The parameters are straightforward here. No need for additional parameters since the query is direct. Just in